## 리스트 컴프리헨션 (List Comprehension)

우리가 데이터를 처리할 때 가장 흔하게 하는 패턴은 **"기존 리스트를 반복문으로 돌면서, 특정 조건에 맞는 값을 변형하여 새로운 리스트에 담는 것"**이다.

### 고전적인 방식 (Classic Style)
전통적인 방식은 빈 리스트를 만들고 `append`를 사용한다.

In [2]:
# 0부터 9까지 숫자 중 짝수만 골라 3을 곱한 리스트 만들기
numbers = range(10)
result = []

for n in numbers:
    if n % 2 == 0:
        result.append(n * 3)

# 결과: [0, 6, 12, 18, 24]
print(result)

[0, 6, 12, 18, 24]


### 파이썬다운 방식 (Pythonic Style)

리스트 컴프리헨션을 사용하면 위 코드를 수학의 '조건제시법'처럼 한 줄로 기술할 수 있다.

In [3]:
# [표현식 for 항목 in 반복가능객체 if 조건]
result = [n * 3 for n in numbers if n % 2 == 0]

print(result) # [0, 6, 12, 18, 24]

[0, 6, 12, 18, 24]


이 방식은 코드가 짧아질 뿐만 아니라, 내부적으로 C로 최적화된 속도로 실행되기 때문에 `append`를 사용하는 `for` 문보다 **처리 속도가 더 빠르다.** 데이터 전처리 로직이 단순하다면 적극적으로 컴프리헨션을 사용해야 한다.

## 제너레이터 (Generator): 메모리를 아끼는 기술

리스트 컴프리헨션은 훌륭하지만 치명적인 단점이 있다. **만들어진 모든 데이터를 즉시 메모리에 올린다**는 점이다.

만약 우리가 처리해야 할 데이터가 100만 개가 아니라 **100억 개** 라면 어떻게 될까? 리스트 컴프리헨션으로 100억 개의 리스트를 만드는 순간, 컴퓨터의 RAM이 가득 차서 프로그램이 비정상적으로 종료되어 버릴 것이다(Memory Error).

이때 **제너레이터 표현식(Generator Expression)** 을 사용한다. 문법은 아주 간단하다. 대괄호 `[]`를 소괄호 `()`로 바꾸기만 하면 된다.

In [ ]:
# 리스트 컴프리헨션: 실제 생성 (메모리 많이 씀)
list_data = [n * 3 for n in range(100000000)]

In [6]:
# 제너레이터 표현식: 생성할 준비만 함 (메모리 거의 안 씀)
gen_data = (n * 3 for n in range(100000000))

### JIT (Just-In-Time) 생산 방식

산업공학에서 재고 비용을 줄이기 위해 JIT 생산 방식을 쓰듯, 제너레이터는 **데이터를 미리 만들어두지 않는다.** 대신 `next()`를 호출하거나 `for` 문이 돌아갈 때마다 **"그때그때 하나씩 계산해서"** 던져준다.

이를 **지연 평가(Lazy Evaluation)** 라고 한다.

In [7]:
import sys

# 리스트: 모든 데이터를 메모리에 적재
big_list = [i for i in range(10000)]
print(sys.getsizeof(big_list))  # 약 87,616 bytes (크기에 비례해 커짐)

85176


In [9]:
# 제너레이터: 생성 규칙만 기억
big_gen = (i for i in range(10000))
print(sys.getsizeof(big_gen))   
# 약 192 bytes (데이터가 100억 개라도 크기 고정) 
# 파이썬 버전에 따라 제너레이터 객체의 크기가 다를 수 있음

192


## 언제 무엇을 써야 할까?

* **List Comprehension `[]`:**
    * 데이터 크기가 작을 때 (메모리에 올려도 부담 없을 때)
    * 데이터를 반복해서 재사용해야 할 때
    * 인덱싱(`data[3]`)이나 슬라이싱이 필요할 때
* **Generator `()`:**
    * 데이터 크기가 매우 클 때 (대용량 로그 파일 분석 등)
    * 데이터를 한 번만 훑고 버릴 때 (One-pass)
    * 무한한 데이터 스트림을 처리할 때

## [심화] 제너레이터는 리스트보다 무조건 좋을까? (속도 vs 메모리)

여기서 한 가지 공학적인 의문이 들어야 한다. "**메모리를 획기적으로 줄여준다면, 그 대가로 속도를 잃는 것은 아닐까?**" 

세상에 공짜 점심은 없기 때문이다.

결론부터 말하면 **"순수 계산 속도는 리스트가 미세하게 빠르지만, 대용량 데이터 환경에서는 제너레이터가 더 효율적이다"** 라고 할 수 있다. 이 미묘한 차이를 이해하기 위해 '생성 시간'과 '순회 시간'을 구분해 보자.

아래 코드는 5,000만 개의 데이터를 처리할 때의 메모리와 시간 차이를 보여준다.

In [10]:
import sys
import time

# 데이터 크기: 5천만 건
SIZE = 50_000_000

print(f"--- 데이터 개수: {SIZE:,}개 비교 ---")

--- 데이터 개수: 50,000,000개 비교 ---


In [11]:
# 1. 리스트 컴프리헨션
start = time.time()
list_data = [i * 2 for i in range(SIZE)] # 이미 여기서 시간 소요
init_time = time.time() - start

sum_val = 0
for n in list_data:
    sum_val += n
proc_time = time.time() - start

# 메모리 사용량 (MB)
mem_usage = sys.getsizeof(list_data) / 1024 / 1024

print(f"[List] 초기화: {init_time:.4f}초 | 전체: {proc_time:.4f}초")
print(f"[List] 메모리: {mem_usage:.2f} MB")

print("-" * 30)

[List] 초기화: 3.6013초 | 전체: 6.4270초
[List] 메모리: 392.86 MB
------------------------------


In [12]:
# 2. 제너레이터 표현식
start = time.time()
gen_data = (i * 2 for i in range(SIZE)) # 즉시 리턴
init_time = time.time() - start

sum_val = 0
for n in gen_data:
    sum_val += n
proc_time = time.time() - start

mem_usage = sys.getsizeof(gen_data) # Bytes 단위

print(f"[Gen ] 초기화: {init_time:.4f}초 | 전체: {proc_time:.4f}초")
print(f"[Gen ] 메모리: {mem_usage} Bytes")

[Gen ] 초기화: 0.0001초 | 전체: 4.4972초
[Gen ] 메모리: 200 Bytes



* 데이터가 메모리에 충분히 들어갈 만큼 작고, 반복해서 써야 한다면 $\rightarrow$ **List**
* 데이터가 매우 크거나, 한 번만 읽고(One-pass) 버릴 것이라면 $\rightarrow$ **Generator**